# Bond math

Rates are decimals (`0.05` = 5%). Maturity is in years; `k` is the number of coupon payments per year.


In [ ]:
import numpy as np
import pandas as pd


## Price, duration and convexity

A five-year bond paying 5%, with a 4.5% yield and semiannual coupons. Duration is in years. The dollar columns use price / 100; DV01 is modified duration × price / 10,000.


In [ ]:
def coupon_freq_mat(c, T, ytm, fv=100, k=1):
    if k <= 0:
        raise ValueError("k must be strictly positive.")
    if T <= 0:
        raise ValueError("T must be strictly positive.")

    periods = int(round(k * T))
    coupon = c * fv / k
    payment_times = np.arange(1, periods + 1)
    cashflows = np.full(periods, coupon, dtype=float)
    cashflows[-1] += fv

    disc = (1 + ytm / k) ** payment_times
    pv_cashflows = cashflows / disc
    price = pv_cashflows.sum()

    times_years = payment_times / k
    d_mac = (times_years * pv_cashflows).sum() / price
    d_mod = d_mac / (1 + ytm / k)
    conv = (times_years * (times_years + 1 / k) * pv_cashflows).sum()
    conv /= price * (1 + ytm / k) ** 2

    summary = pd.DataFrame([{
        'Coupon': coupon,
        'DCF': price,
        'Macaulay duration': d_mac,
        'Modified duration': d_mod,
        'Dollar duration': d_mod * price / 100,
        'Convexity': conv,
        'Dollar convexity': conv * price / 100,
    }])
    return summary


In [ ]:
coupon_freq_mat(c=0.05, T=5, ytm=0.045, fv=100, k=2)


## Amortization

The carrying value moves toward face value. The last balance is before principal repayment.


In [ ]:
def amortization_table(c, T, ytm, fv=100, k=1):
    if k <= 0:
        raise ValueError("k must be strictly positive.")
    if T <= 0:
        raise ValueError("T must be strictly positive.")

    periods = int(round(k * T))
    coupon = c * fv / k

    cashflows = np.full(periods, coupon, dtype=float)
    cashflows[-1] += fv
    period_rate = ytm / k
    discount = (1 + period_rate) ** np.arange(1, periods + 1)
    current_balance = (cashflows / discount).sum()
    rows = []
    for period in range(1, periods + 1):
        interest_income = current_balance * period_rate
        amortization = interest_income - coupon
        ending_balance = current_balance + amortization
        rows.append({
            'Period': period,
            'Beginning balance': current_balance,
            'Coupon payment': coupon,
            'Interest at YTM': interest_income,
            'Amortization': amortization,
            'Ending balance': ending_balance,
        })
        current_balance = ending_balance

    return pd.DataFrame(rows)


In [ ]:
amortization_table(c=0.05, T=5, ytm=0.045, fv=100, k=2)


## Continuous discounting

Annual coupons, discounted with $e^{-yt}$. Use whole-year maturities.


In [ ]:
def coupon_continuous(c, T, ytm, fv=100):
    if T <= 0:
        raise ValueError("T must be strictly positive.")

    periods = int(round(T))
    coupon = c * fv
    times = np.arange(1, periods + 1)
    cashflows = np.full(periods, coupon, dtype=float)
    cashflows[-1] += fv

    pv_cashflows = cashflows * np.exp(-ytm * times)
    price = pv_cashflows.sum()
    macaulay_duration = (times * pv_cashflows).sum() / price
    return {
        'Coupon': coupon,
        'DCF': price,
        'Macaulay duration': macaulay_duration,
    }


In [ ]:
coupon_continuous(c=0.05, T=5, ytm=0.045)


## Discounted cash flows

Use one yield or a rate for each payment. Payment times default to 1, 2, 3, … years.


In [ ]:
def dcf(cf, ytm=None, r=None, times=None):
    cf = np.asarray(cf, dtype=float)
    if r is None:
        if ytm is None:
            raise ValueError("Provide either ytm or r.")
        r = ytm
    if times is None:
        times = np.arange(1, len(cf) + 1)
    return np.sum(cf / (1 + np.asarray(r)) ** np.asarray(times))


In [ ]:
cashflows = np.array([5.0, 5.0, 105.0])
pd.Series({
    "Flat yield": dcf(cashflows, ytm=0.045),
    "Spot rates": dcf(cashflows, r=np.array([0.04, 0.045, 0.05])),
})
